In [0]:
%sql
-- Pergunta 1: evolução anual de conexões e potência instalada
SELECT t.ano,
       COUNT(*) AS qtd_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_tempo t ON f.sk_tempo = t.sk_tempo
WHERE t.sk_tempo <> -1
GROUP BY t.ano
ORDER BY t.ano

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 2a: potência e porte médio por estado
SELECT l.SigUF,
       COUNT(*) AS qtd_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw,
       ROUND(AVG(f.potencia_instalada_kw), 1) AS kw_por_conexao
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_localidade l ON f.sk_localidade = l.sk_localidade
GROUP BY l.SigUF
ORDER BY potencia_mw DESC

In [0]:
%sql
-- Pergunta 2b: 10 municípios com maior potência instalada
SELECT l.NomMunicipio,
       l.SigUF,
       COUNT(*) AS qtd_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw,
       ROUND(AVG(f.potencia_instalada_kw), 1) AS kw_por_conexao
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_localidade l ON f.sk_localidade = l.sk_localidade
GROUP BY l.NomMunicipio, l.SigUF
ORDER BY potencia_mw DESC
LIMIT 10

In [0]:
%sql
-- Pergunta 3: distribuição das conexões entre distribuidoras
SELECT d.SigAgente,
       COUNT(*) AS qtd_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_conexoes
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_distribuidora d ON f.sk_distribuidora = d.sk_distribuidora
GROUP BY d.SigAgente
ORDER BY qtd_conexoes DESC
LIMIT 15

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 4a: adoção por classe de consumo
SELECT TRIM(c.DscClasseConsumo) AS classe,
       COUNT(*) AS qtd_conexoes,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw,
       ROUND(SUM(f.potencia_instalada_kw) * 100.0 / SUM(SUM(f.potencia_instalada_kw)) OVER (), 1) AS pct_potencia,
       ROUND(AVG(f.potencia_instalada_kw), 1) AS kw_por_conexao
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_classificacao_consumo c ON f.sk_classificacao_consumo = c.sk_classificacao_consumo
GROUP BY TRIM(c.DscClasseConsumo)
ORDER BY qtd_conexoes DESC

In [0]:
%sql
-- Pergunta 4b: micro x minigeração
SELECT fo.DscPorte,
       COUNT(*) AS qtd_conexoes,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw,
       ROUND(SUM(f.potencia_instalada_kw) * 100.0 / SUM(SUM(f.potencia_instalada_kw)) OVER (), 1) AS pct_potencia
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_fonte fo ON f.sk_fonte = fo.sk_fonte
GROUP BY fo.DscPorte

In [0]:
%sql
-- Pergunta 5: participação de cada modalidade
SELECT COALESCE(m.DscModalidadeHabilitado, 'Não informado') AS modalidade,
       COUNT(*) AS qtd_conexoes,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_conexoes,
       ROUND(SUM(f.potencia_instalada_kw) / 1000, 1) AS potencia_mw,
       ROUND(SUM(f.potencia_instalada_kw) * 100.0 / SUM(SUM(f.potencia_instalada_kw)) OVER (), 1) AS pct_potencia,
       ROUND(AVG(f.potencia_instalada_kw), 1) AS kw_por_conexao
FROM mvp_mmgd.gold.f_conexao_mmgd f
JOIN mvp_mmgd.gold.d_modalidade m ON f.sk_modalidade = m.sk_modalidade
GROUP BY COALESCE(m.DscModalidadeHabilitado, 'Não informado')
ORDER BY qtd_conexoes DESC

In [0]:
%sql
-- Pergunta 6a: fabricantes de módulos
SELECT d.grupo_fabricante AS fabricante_modulo,
       COUNT(*) AS qtd_sistemas,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_sistemas
FROM mvp_mmgd.gold.f_tecnica_fv t
JOIN mvp_mmgd.gold.d_fabricante d ON t.sk_fabricante_modulo = d.sk_fabricante
GROUP BY d.grupo_fabricante
ORDER BY qtd_sistemas DESC
LIMIT 12

In [0]:
%sql
-- Pergunta 6b: fabricantes de inversores
SELECT d.grupo_fabricante AS fabricante_inversor,
       COUNT(*) AS qtd_sistemas,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_sistemas
FROM mvp_mmgd.gold.f_tecnica_fv t
JOIN mvp_mmgd.gold.d_fabricante d ON t.sk_fabricante_inversor = d.sk_fabricante
GROUP BY d.grupo_fabricante
ORDER BY qtd_sistemas DESC
LIMIT 12

In [0]:
%sql
-- Pergunta 7: fator de dimensionamento (considerando só valores entre 0,5 e 3,0)
SELECT COUNT(*) AS qtd_sistemas,
       ROUND(PERCENTILE(fator_dimensionamento, 0.5), 2) AS mediana,
       ROUND(AVG(fator_dimensionamento), 2) AS media,
       ROUND(PERCENTILE(fator_dimensionamento, 0.25), 2) AS p25,
       ROUND(PERCENTILE(fator_dimensionamento, 0.75), 2) AS p75
FROM mvp_mmgd.gold.f_tecnica_fv
WHERE fator_dimensionamento BETWEEN 0.5 AND 3.0